In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import itertools

from smt.sampling_methods import LHS
from smt.surrogate_models import MixIntKernelType
from smt.surrogate_models.krg import KRG

from comcatalyst.context.cases import case_01, case_02, case_03, case_03_bis_2d, case_04
from comcatalyst.xprs.xpr_smt import XprSmt
from comcatalyst.xprs.visualizations import (
    plot_1d_ground_truth, plot_1d_gp_quant_by_qual, 
    plot_contour_quant_by_qual,
    plot_3d_quant_by_qual,
    plot_gp_at_iter,
    animate_gp_at_iters,
    plot_convergence
)

In [ ]:
# context
case = case_04()

# hyperparameters
seed = 4 
seed_ydoe_offset = 1000
n_init = 2
n_iter_bo = 28
n_start = 20
sampling_method = LHS
sampling_criterion = "ese"
kernel = MixIntKernelType.CONT_RELAX
bo_criterion = "EI"
sigma_noise = 0.1
qei="KBLB"
prune_every=6
beta=1.96
n_grid_cont=10

In [ ]:
# True max
x_true_max, y_true_max, X_grid, Y_grid = XprSmt.calc_true_max(
    fn_gp_objective=case.fn_gp_objective, 
    vars_metadata=case.vars_metadata, 
    z=case.z, 
    n_grid=1000
)


In [ ]:
print(f"x_true_max = {case.vars_metadata.str_values(x_true_max)}")
print(f"y_true_max = {y_true_max:.12f}")

In [ ]:
# check répartition maxima
categ_combos = list(itertools.product(
    case.vars_metadata[0].support, # qual1
    case.vars_metadata[1].support, # qual2
))

for combo in categ_combos:
    mask = (
        (X_grid[:, 0] == case.vars_metadata[0].support.index(combo[0])) &
        (X_grid[:, 1] == case.vars_metadata[1].support.index(combo[1]))
    )
    y_combo = Y_grid[mask]
    combo_label = f"{combo[0]}/{combo[1]}"
    print(
        f"{combo_label:<8} :  "
        f"max ={y_combo.max():6.3f}   "
        f"min ={y_combo.min():7.3f}   "
        f"mean ={y_combo.mean():6.3f}   "
        f"% > 0.8*max ={100*(y_combo > 0.8*Y_grid.max()).mean():5.1f}%"
    )

In [ ]:
# plot true max 1d
plot_1d_ground_truth(
    fn_gp_objective=case.fn_gp_objective,
    vars_metadata=case.vars_metadata,
    z=case.z,
    quant_fixed_vals=(
        (2, x_true_max[2]), 
        (3, x_true_max[3]),
       # (4, x_true_max[4])
    ),
    n_points=1000,
    figsize=(12,9),
    ncol_legend=2,
    fontsize=18
)

In [ ]:
# plot true max 3d
plot_3d_quant_by_qual(
    metric='fn',
    vars_metadata=case.vars_metadata,
    quant_to_variate_idx_1=2,
    quant_to_variate_idx_2=3,
    # quant_fixed_vals=[(4, 0.0505)],
    qual_filter=None,
    fn_gp_objective=case.fn_gp_objective,
    z=case.z,
    x_true_max=x_true_max,
    y_true_max=y_true_max,
    n_grid=30,
    n_cols=4,
    fontsize=10,
    title=""
)

In [ ]:
# plot true max contour
plot_contour_quant_by_qual(
    metric='fn',
    vars_metadata=case.vars_metadata,
    quant_to_variate_idx_1=2,
    quant_to_variate_idx_2=3,
    # quant_fixed_vals=[(4, 0.0505)],
    qual_filter=None,
    fn_gp_objective=case.fn_gp_objective,
    z=case.z,
    x_true_max=x_true_max,
    y_true_max=y_true_max,
    n_grid=30,
    n_cols=4,
    figsize=(11,8),
    fontsize=16,
    title=""
)

In [ ]:
# EGO
x_init, design_space = XprSmt.calc_x_init_with_sampling_method(
    vars_metadata=case.vars_metadata,
    n_init=n_init,
    sampling_method=sampling_method,
    sampling_criterion=sampling_criterion,
    seed=seed
)

y_init = case.fn_gp_objective(
    vars_metadata=case.vars_metadata,
    x=x_init, 
    z=case.z,
    sigma_noise=sigma_noise, 
    rng=np.random.default_rng(seed+seed_ydoe_offset)
)

sm = KRG(
    design_space=design_space,
    categorical_kernel=kernel,
    print_global=False,
    eval_noise=True,
    hyper_opt="Cobyla"
)

res_ego = XprSmt.run_ego(
    fn_gp_objective=case.fn_gp_objective,
    vars_metadata=case.vars_metadata,
    design_space=design_space,
    x_init=x_init,
    y_init=y_init,
    z=case.z,
    n_iter_bo=n_iter_bo,
    kernel=kernel,
    bo_criterion=bo_criterion,
    qei=qei,
    sigma_noise=sigma_noise,
    n_start=n_start,
    seed=seed,
    sm=sm
)

In [ ]:
# EGO results details
print(f"--- BO x_data ---\n{res_ego.x_data}")
print(f"\n--- BO y_data ---\n{res_ego.y_data}")
print(f"\n--- BO y_true_at_x_data ---\n{res_ego.y_true_at_x_data}")

print("\n--- Results ---")
print(" "*2 + f"x_opt = {case.vars_metadata.str_values(res_ego.x_opt)}")
print(" "*2 + f"y_opt = {res_ego.y_opt:.4f}")
print(" "*2 + f"y_true_at_x_opt = {res_ego.y_true_at_x_opt:.4f}")

In [ ]:
# check repartition qualitative variables
for q1 in range(len(case.vars_metadata[0].support)): 
    for q2 in range(len(case.vars_metadata[1].support)): 
        mask = (res_ego.x_data[:n_init,0]==q1) & (res_ego.x_data[:n_init,1]==q2)
        print(f"qual1={q1} qual2={q2}: {mask.sum()} points")

In [ ]:
# EGO b&b
x_init, design_space = XprSmt.calc_x_init_with_sampling_method(
    vars_metadata=case.vars_metadata,
    n_init=n_init,
    sampling_method=sampling_method,
    sampling_criterion=sampling_criterion,
    seed=seed
)

y_init = case.fn_gp_objective(
    vars_metadata=case.vars_metadata,
    x=x_init, 
    z=case.z,
    sigma_noise=sigma_noise, 
    rng=np.random.default_rng(seed+seed_ydoe_offset)
)

sm = KRG(
    design_space=design_space,
    categorical_kernel=kernel,
    print_global=False,
    eval_noise=True,
    hyper_opt="Cobyla"
)

res_ego_bb = XprSmt.run_ego_with_bb(
    fn_gp_objective_os=case.fn_gp_objective,
    vars_metadata_os=case.vars_metadata,
    z=case.z,
    design_space_os=design_space,
    x_init_os=x_init,
    y_init_os=y_init,
    n_iter_bo=n_iter_bo,
    kernel=kernel,
    bo_criterion=bo_criterion,
    qei=qei,
    sigma_noise=sigma_noise,
    n_start=n_start,
    seed=seed,
    prune_every=prune_every,
    beta=beta,
    n_grid_cont=n_grid_cont,
    sm=sm
)

In [ ]:
# EGO b&b results details
print(f"--- BO x_data ---\n{res_ego_bb.x_data}")
print(f"\n--- BO y_data ---\n{res_ego_bb.y_data}")
print(f"\n--- BO y_true_at_x_data ---\n{res_ego_bb.y_true_at_x_data}")

print("\n--- Results ---")
print(" "*2 + f"x_opt = {case.vars_metadata.str_values(res_ego_bb.x_opt)}")
print(" "*2 + f"y_opt = {res_ego_bb.y_opt:.4f}")
print(" "*2 + f"y_true_at_x_opt = {res_ego_bb.y_true_at_x_opt:.4f}")

In [ ]:
# plot convergence
plot_convergence(
    y_data=res_ego.y_true_at_x_data,
    n_init=n_init,
)

In [ ]:
# plot 1d GP quant1 by qual
plot_1d_gp_quant_by_qual(
    sm=res_ego.sm,
    vars_metadata=case.vars_metadata,
    x_opt=res_ego.x_opt,
    quant_to_variate_idx=2,
    reverse_optim=True,
    qual_filter=[(0, ["A", "B"]), (1, ["alpha", "beta"])],
    x_data=res_ego.x_data,
    y_data=res_ego.y_data,
    n_init=n_init,
    y_opt=res_ego.y_opt,
    x_true_max=x_true_max,
    y_true_max=y_true_max,
    figsize=(9,5)
)

In [ ]:
# plot 1d GP quant2 by qual
plot_1d_gp_quant_by_qual(
    sm=res_ego.sm,
    vars_metadata=case.vars_metadata,
    x_opt=res_ego.x_opt,
    quant_to_variate_idx=3,
    reverse_optim=True,
    qual_filter=[(0, ["A", "B"]), (1, ["alpha", "beta"])],
    x_data=res_ego.x_data,
    y_data=res_ego.y_data,
    n_init=n_init,
    y_opt=res_ego.y_opt,
    x_true_max=x_true_max,
    y_true_max=y_true_max,
    figsize=(9,5)    
)

In [ ]:
# plot GP contour level
plot_contour_quant_by_qual(
    metric='mu',
    vars_metadata=case.vars_metadata,
    quant_to_variate_idx_1=2,
    quant_to_variate_idx_2=3,
    qual_filter=None,
    sm=res_ego.sm,
    reverse_optim=True,
    x_data=res_ego.x_data,
    y_data=res_ego.y_data,
    n_init=n_init,
    x_opt=res_ego.x_opt,
    y_opt=res_ego.y_opt,
    x_true_max=x_true_max,
    y_true_max=y_true_max,
    n_grid=30,
    n_cols=4,
    figsize=(7,5)
)

In [ ]:
# plot GP 3d
plot_3d_quant_by_qual(
    metric='mu',
    vars_metadata=case.vars_metadata,
    quant_to_variate_idx_1=2,
    quant_to_variate_idx_2=3,
    qual_filter=None,
    sm=res_ego.sm,
    reverse_optim=True,
    x_data=res_ego.x_data,
    y_data=res_ego.y_data,
    n_init=n_init,
    x_opt=res_ego.x_opt,
    y_opt=res_ego.y_opt,
    x_true_max=x_true_max,
    y_true_max=y_true_max,
    n_grid=30,
    n_cols=4
)

In [ ]:
# plot contour gp at iter
plot_gp_at_iter(
    plot_type='contour',
    metric='mu',
    result_ego=res_ego,
    vars_metadata=case.vars_metadata,
    n_init=n_init,
    iter_bo=0,
    quant_to_variate_idx_1=2,
    quant_to_variate_idx_2=3,
    quant_fixed_vals=None,
    qual_filter=None,
    n_grid=30,
    n_cols=2,
    figsize=(7,5),
    title=None,
    save_path=None
)

In [ ]:
# plot 3d gp at iter
plot_gp_at_iter(
    plot_type='3d',
    metric='mu',
    result_ego=res_ego,
    vars_metadata=case.vars_metadata,
    n_init=n_init,
    iter_bo=4,
    quant_to_variate_idx_1=2,
    quant_to_variate_idx_2=3,
    quant_fixed_vals=None,
    qual_filter=None,
    n_grid=30,
    n_cols=2,
    figsize=(6,5),
    title=None,
    save_path=None
)

In [ ]:
for metric in ['mu', 'sd', 'ei']:
    animate_gp_at_iters(
        plot_type='contour',
        metric=metric,
        result_ego=res_ego,
        vars_metadata=case.vars_metadata,
        n_init=n_init,
        quant_to_variate_idx_1=2,
        quant_to_variate_idx_2=3,
        quant_fixed_vals=None,
        qual_filter=None,
        n_grid=20,
        n_cols=4,
        figsize=(7,5),
        fps=1,
        save_path=f"../../data/xpr_smt/animations/cas04_sigma01_init10_good_contour_{metric}.mp4",
    )